# 🌾 Maharashtra Crop Recommendation System
## Complete End-to-End Machine Learning Pipeline

**Author:** Agricultural AI System  
**Dataset:** Maharashtra Crop Recommendation Dataset  
**Goal:** Multi-class classification to predict the best crop for given soil and environmental conditions across Maharashtra districts.

---

## 📋 Table of Contents

1. [Library Imports & Setup](#1)
2. [Data Loading & Exploration](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Data Preprocessing](#4)
5. [Feature Engineering](#5)
6. [Model Training - 5 Algorithms](#6)
7. [Model Evaluation & Comparison](#7)
8. [Cross Validation](#8)
9. [Confusion Matrix Analysis](#9)
10. [Feature Importance](#10)
11. [GridSearchCV Hyperparameter Tuning](#11)
12. [Best Model Selection & Saving](#12)
13. [Prediction Function](#13)
14. [Visualizations & Charts](#14)
15. [Summary Report](#15)

---
## 1. 📦 Library Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
import joblib

# Style
plt.rcParams.update({
    "figure.facecolor": "#f8fff8",
    "axes.facecolor": "#f0f8f0",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "font.size": 11,
})
sns.set_theme(style="whitegrid", palette="Set2")
print("✅ All libraries imported successfully!")

---
## 2. 📊 Data Loading & Exploration

In [ ]:
# Load dataset
df = pd.read_csv("dataset/crop_recommendation_maharashtra_area_dataset.csv")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)
df.info()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print(f"
Duplicate Rows: {df.duplicated().sum()}")

In [ ]:
print("Unique Districts:", df["district_area_maharashtra"].nunique())
print(df["district_area_maharashtra"].unique())
print("
Unique Crops:", df["crop_label"].nunique())
print(df["crop_label"].unique())

In [ ]:
df.describe().round(2)

---
## 3. 📈 Exploratory Data Analysis (EDA)

In [ ]:
# Crop distribution
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

crop_counts = df["crop_label"].value_counts()
axes[0].barh(crop_counts.index, crop_counts.values,
             color=plt.cm.Set3(np.linspace(0, 1, len(crop_counts))))
axes[0].set_title("Crop Distribution", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Count")

axes[1].pie(crop_counts.values, labels=crop_counts.index,
            autopct="%1.1f%%", colors=plt.cm.Set3(np.linspace(0, 1, len(crop_counts))),
            startangle=140, textprops={"fontsize": 8})
axes[1].set_title("Crop Proportion", fontsize=14, fontweight="bold")

plt.suptitle("Maharashtra Crop Dataset Distribution", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# District distribution
fig, ax = plt.subplots(figsize=(14, 6))
dist_counts = df["district_area_maharashtra"].value_counts()
bars = ax.bar(dist_counts.index, dist_counts.values,
              color=sns.color_palette("viridis", len(dist_counts)))
ax.set_title("Records per District", fontsize=14, fontweight="bold")
ax.set_xlabel("District")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 7))
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn",
            mask=mask, ax=ax, vmin=-1, vmax=1,
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Box plots - soil nutrients by crop
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, col in zip(axes.flat, ["nitrogen", "phosphorus", "potassium", "ph"]):
    df.boxplot(column=col, by="crop_label", ax=ax, rot=90,
               patch_artist=True, boxprops=dict(facecolor="#c8e6c9"),
               medianprops=dict(color="red", linewidth=2))
    ax.set_title(f"{col.capitalize()} by Crop", fontsize=12, fontweight="bold")
    ax.set_xlabel("Crop")
plt.suptitle("Soil Nutrients Distribution by Crop", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot of numeric features (sample)
sample_df = df[df["crop_label"].isin(["rice","cotton","grapes","maize","banana"])].copy()
g = sns.pairplot(sample_df[["nitrogen","phosphorus","temperature","rainfall","crop_label"]],
                 hue="crop_label", corner=True, plot_kws={"alpha": 0.6})
g.figure.suptitle("Pair Plot: Sample Crops", y=1.02, fontsize=14)
plt.show()

---
## 4. ⚙️ Data Preprocessing

In [ ]:
# Drop missing values
df_clean = df.copy()
df_clean.dropna(inplace=True)
print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning : {len(df_clean)}")

# Label encoders
district_encoder = LabelEncoder()
crop_encoder     = LabelEncoder()

df_clean["district_encoded"] = district_encoder.fit_transform(df_clean["district_area_maharashtra"])
df_clean["crop_encoded"]     = crop_encoder.fit_transform(df_clean["crop_label"])

print("
District Encoding:")
for i, d in enumerate(district_encoder.classes_):
    print(f"  {i}: {d}")
print("
Crop Encoding:")
for i, c in enumerate(crop_encoder.classes_):
    print(f"  {i}: {c}")

In [ ]:
# Feature matrix and target
features = ["district_encoded", "nitrogen", "phosphorus", "potassium",
            "temperature", "humidity", "ph", "rainfall"]
feature_labels = ["District", "Nitrogen", "Phosphorus", "Potassium",
                  "Temperature", "Humidity", "pH", "Rainfall"]

X = df_clean[features].values
y = df_clean["crop_encoded"].values

print(f"Feature matrix X: {X.shape}")
print(f"Target vector  y: {y.shape}")
print(f"Classes        : {len(np.unique(y))}")

In [ ]:
# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled Feature Statistics:")
pd.DataFrame(X_scaled, columns=feature_labels).describe().round(3)

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")
print(f"Train ratio   : {len(X_train)/len(X_scaled)*100:.1f}%")
print(f"Test ratio    : {len(X_test)/len(X_scaled)*100:.1f}%")

---
## 5. 🔧 Feature Engineering

In [ ]:
# Add interaction features
df_fe = pd.DataFrame(X_scaled, columns=feature_labels)
df_fe["N_P_ratio"]   = df_fe["Nitrogen"] / (df_fe["Phosphorus"] + 0.001)
df_fe["N_K_ratio"]   = df_fe["Nitrogen"] / (df_fe["Potassium"] + 0.001)
df_fe["NPK_total"]   = df_fe["Nitrogen"] + df_fe["Phosphorus"] + df_fe["Potassium"]
df_fe["Temp_Humid"]  = df_fe["Temperature"] * df_fe["Humidity"]

print("Engineered feature columns:")
print(df_fe.columns.tolist())
df_fe.head()

---
## 6. 🤖 Model Training — 5 Algorithms

In [ ]:
models = {
    "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "Decision Tree":       DecisionTreeClassifier(max_depth=15, random_state=42),
    "SVM":                 SVC(probability=True, kernel="rbf", C=10, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
}

# Train all models
trained = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    trained[name] = m
    print(f"✅ Trained: {name}")

---
## 7. 📊 Model Evaluation & Comparison

In [ ]:
results = {}
for name, m in trained.items():
    y_pred = m.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {"accuracy": round(acc*100, 2), "model": m, "y_pred": y_pred}

# Print comparison
print(f"{"Model":<25} {"Test Accuracy":>15}")
print("-" * 42)
for name, r in sorted(results.items(), key=lambda x: -x[1]["accuracy"]):
    mark = "🏆" if r["accuracy"] == max(v["accuracy"] for v in results.values()) else "  "
    print(f"{mark} {name:<23} {r["accuracy"]:>14.2f}%")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 5))
names = list(results.keys())
accs  = [results[n]["accuracy"] for n in names]
colors = ["#2e7d32" if a == max(accs) else "#90caf9" for a in accs]
bars = ax.bar(names, accs, color=colors, edgecolor="white", linewidth=1.5, width=0.55)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.1,
            f"{val}%", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_title("Model Accuracy Comparison", fontsize=14, fontweight="bold")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, max(accs)+3)
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report for Best Model
best_name = max(results, key=lambda k: results[k]["accuracy"])
best_model_obj = trained[best_name]
y_pred_best = results[best_name]["y_pred"]

print(f"Classification Report — {best_name}")
print("=" * 80)
print(classification_report(y_test, y_pred_best, target_names=crop_encoder.classes_, zero_division=0))

---
## 8. 🔁 Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, m in trained.items():
    cv_scores = cross_val_score(m, X_scaled, y, cv=skf, scoring="accuracy", n_jobs=-1)
    cv_results[name] = cv_scores
    print(f"{name:<25} CV: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

In [ ]:
# Box plot of CV scores
fig, ax = plt.subplots(figsize=(12, 6))
data_plot = [cv_results[n]*100 for n in models.keys()]
bp = ax.boxplot(data_plot, labels=list(models.keys()), patch_artist=True, notch=True)
colors = ["#a5d6a7","#90caf9","#f48fb1","#ffe082","#ce93d8"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
ax.set_title("Cross-Validation Score Distribution (5-Fold)", fontsize=14, fontweight="bold")
ax.set_ylabel("Accuracy (%)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

---
## 9. 🔲 Confusion Matrix Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flat
model_list = list(trained.items())

for i, (name, m) in enumerate(model_list):
    cm = confusion_matrix(y_test, results[name]["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=crop_encoder.classes_)
    disp.plot(ax=axes[i], colorbar=False, xticks_rotation=90)
    axes[i].set_title(f"{name}", fontsize=11, fontweight="bold")
    axes[i].tick_params(labelsize=7)

# Hide unused subplot
axes[-1].set_visible(False)
plt.suptitle("Confusion Matrices — All Models", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 10. ⭐ Feature Importance

In [ ]:
# Get feature importances from tree-based models
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, model_name in zip(axes, ["Random Forest", "Decision Tree", "Gradient Boosting"]):
    m = trained[model_name]
    imp = m.feature_importances_
    idx = np.argsort(imp)
    ax.barh([feature_labels[i] for i in idx], imp[idx],
            color=plt.cm.viridis(np.linspace(0.3, 0.9, len(imp))))
    ax.set_title(f"Feature Importance
{model_name}", fontweight="bold")
    ax.set_xlabel("Importance Score")
plt.suptitle("Feature Importance Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 11. 🔍 GridSearchCV — Hyperparameter Tuning

In [ ]:
# GridSearchCV for Random Forest (reduced grid for speed)
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf, cv=3, scoring="accuracy", n_jobs=-1, verbose=1)

print("Running GridSearchCV for Random Forest...")
rf_grid.fit(X_train, y_train)
print(f"Best Parameters : {rf_grid.best_params_}")
print(f"Best CV Score   : {rf_grid.best_score_*100:.2f}%")
rf_tuned = rf_grid.best_estimator_

In [ ]:
# GridSearchCV for Gradient Boosting
param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
}
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid_gb, cv=3, scoring="accuracy", n_jobs=-1, verbose=1)

print("Running GridSearchCV for Gradient Boosting...")
gb_grid.fit(X_train, y_train)
print(f"Best Parameters : {gb_grid.best_params_}")
print(f"Best CV Score   : {gb_grid.best_score_*100:.2f}%")
gb_tuned = gb_grid.best_estimator_

---
## 12. 💾 Best Model Selection & Saving

In [ ]:
# All models including tuned ones
all_models = {**trained, "RF Tuned": rf_tuned, "GB Tuned": gb_tuned}
final_results = {}

for name, m in all_models.items():
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    final_results[name] = acc
    
print("Final Model Comparison:")
for name, acc in sorted(final_results.items(), key=lambda x: -x[1]):
    print(f"  {name:<30} {acc*100:.2f}%")
    
best_final_name = max(final_results, key=final_results.get)
best_final_model = all_models[best_final_name]
print(f"
🏆 Overall Best: {best_final_name} ({final_results[best_final_name]*100:.2f}%)")

In [ ]:
# Save model
joblib.dump(best_final_model, "crop_model.pkl")

# Build and save crop_info
crop_info = {
    "rice":        {"season":"Kharif","water":"High","fertilizer":"Urea + DAP","desc":"Staple cereal crop requiring flooded fields."},
    "cotton":      {"season":"Kharif","water":"Moderate","fertilizer":"NPK 20:20:0","desc":"Major cash crop of Vidarbha region."},
    "grapes":      {"season":"Rabi","water":"Moderate","fertilizer":"Nitrogen-rich","desc":"Premium horticultural crop of Nashik."},
    "banana":      {"season":"Perennial","water":"High","fertilizer":"Potassium-rich","desc":"Tropical fruit needing warm temperatures."},
    "mango":       {"season":"Summer","water":"Low","fertilizer":"NPK 10:26:26","desc":"King of fruits, grown in Konkan."},
    "orange":      {"season":"Rabi","water":"Moderate","fertilizer":"Boron + Zinc","desc":"Nagpur orange — world-famous citrus."},
    "pomegranate": {"season":"Kharif","water":"Low","fertilizer":"Micronutrient mix","desc":"Drought-tolerant fruit of Solapur."},
    "coconut":     {"season":"Perennial","water":"High","fertilizer":"NPK + FYM","desc":"Coastal crop of Konkan region."},
    "papaya":      {"season":"Kharif","water":"Moderate","fertilizer":"Nitrogen-rich","desc":"Fast-growing tropical fruit."},
    "watermelon":  {"season":"Summer","water":"Moderate","fertilizer":"Phosphorus-rich","desc":"Summer crop for sandy-loam soils."},
    "muskmelon":   {"season":"Summer","water":"Moderate","fertilizer":"NPK 19:19:19","desc":"Warm-season cucurbit."},
    "maize":       {"season":"Kharif","water":"Moderate","fertilizer":"Urea + SSP","desc":"Versatile cereal crop."},
    "chickpea":    {"season":"Rabi","water":"Low","fertilizer":"SSP + Rhizobium","desc":"Important pulse of Marathwada."},
    "pigeonpeas":  {"season":"Kharif","water":"Low","fertilizer":"Phosphorus-rich","desc":"Tur dal of Maharashtra."},
    "mungbean":    {"season":"Kharif","water":"Low","fertilizer":"Rhizobium","desc":"Short-duration pulse."},
    "blackgram":   {"season":"Kharif","water":"Low","fertilizer":"SSP + Rhizobium","desc":"Urad dal crop."},
    "lentil":      {"season":"Rabi","water":"Low","fertilizer":"DAP + MOP","desc":"Cool-season protein pulse."},
    "kidneybeans": {"season":"Kharif","water":"Moderate","fertilizer":"NPK 60:80:40","desc":"Rajma — protein-rich legume."},
    "mothbeans":   {"season":"Kharif","water":"Low","fertilizer":"Phosphorus","desc":"Drought-hardy legume."},
    "coffee":      {"season":"Kharif","water":"High","fertilizer":"NPK + organic","desc":"Plantation crop."},
    "jute":        {"season":"Kharif","water":"High","fertilizer":"Nitrogen-rich","desc":"Natural fibre crop."},
    "apple":       {"season":"Rabi","water":"Moderate","fertilizer":"Calcium + Boron","desc":"Temperate fruit at high elevations."},
}

joblib.dump({
    "district_encoder":    district_encoder,
    "crop_encoder":        crop_encoder,
    "scaler":              scaler,
    "feature_names":       feature_labels,
    "districts":           district_encoder.classes_.tolist(),
    "crops":               crop_encoder.classes_.tolist(),
    "best_model_name":     best_final_name,
    "crop_info":           crop_info,
    "dataset_stats":       df_clean.describe().round(2).to_dict(),
    "crop_distribution":   df_clean["crop_label"].value_counts().to_dict(),
    "district_distribution": df_clean["district_area_maharashtra"].value_counts().to_dict(),
    "feature_importance":  {k: round(float(v), 4) for k, v in zip(feature_labels,
                              best_final_model.feature_importances_
                              if hasattr(best_final_model, "feature_importances_")
                              else [1/8]*8)},
    "model_results":       {name: {"accuracy": round(acc*100,2), "cv_mean":0, "cv_std":0,
                              "classification_report":{}, "confusion_matrix":[]}
                             for name, acc in final_results.items()},
}, "encoders.pkl")
print("✅ crop_model.pkl & encoders.pkl saved!")

---
## 13. 🔮 Prediction Function

In [ ]:
DISTRICT_AFFINITY = {
    "Nashik":    ["grapes","pomegranate","maize"],
    "Pune":      ["grapes","sugarcane","maize","pomegranate"],
    "Nagpur":    ["orange","cotton","rice"],
    "Kolhapur":  ["sugarcane","rice","banana","coconut"],
    "Amravati":  ["cotton","soybean","orange"],
    "Aurangabad":["cotton","maize","orange","pomegranate"],
    "Solapur":   ["pomegranate","cotton","soybean"],
    "Latur":     ["soybean","cotton","grapes","maize"],
    "Wardha":    ["cotton","orange","rice"],
    "Jalgaon":   ["banana","cotton","maize"],
    "Beed":      ["cotton","sugarcane","banana","pomegranate"],
    "Nanded":    ["soybean","cotton","pigeonpeas"],
    "Akola":     ["cotton","chickpea","soybean"],
    "Dhule":     ["banana","wheat","maize","cotton"],
    "Satara":    ["sugarcane","rice","maize"],
    "Sangli":    ["sugarcane","grapes","banana","cotton"],
    "Osmanabad": ["soybean","cotton","banana"],
    "Parbhani":  ["cotton","pigeonpeas","mungbean"],
    "Bhandara":  ["rice","soybean","cotton"],
    "Ahmednagar":["sugarcane","maize","onion","pomegranate"],
}

def predict_crop(district, nitrogen, phosphorus, potassium,
                 temperature, humidity, ph, rainfall):
    """
    Hybrid crop recommendation using ML + agronomy rules.
    Returns top-5 crop recommendations with confidence %.
    """
    enc   = joblib.load("encoders.pkl")
    model = joblib.load("crop_model.pkl")
    
    d_enc  = enc["district_encoder"].transform([district])[0]
    feats  = np.array([[d_enc, nitrogen, phosphorus, potassium,
                         temperature, humidity, ph, rainfall]])
    scaled = enc["scaler"].transform(feats)
    proba  = model.predict_proba(scaled)[0]
    
    # Boost affinity crops
    boosted = proba.copy()
    for c in DISTRICT_AFFINITY.get(district, []):
        if c in enc["crops"]:
            idx = enc["crops"].index(c)
            boosted[idx] *= 2.5
    
    # Agronomy-based boosts
    for i, crop in enumerate(enc["crops"]):
        info = enc["crop_info"].get(crop, {})
        water = info.get("water", "Moderate")
        if water=="High"     and rainfall>800:  boosted[i]*=1.4
        if water=="Low"      and rainfall<500:  boosted[i]*=1.4
        if water=="Moderate" and 500<=rainfall<=800: boosted[i]*=1.2
        if crop in ["cotton","mothbeans"] and temperature>28: boosted[i]*=1.2
        if crop in ["chickpea","mungbean","blackgram"] and nitrogen<50: boosted[i]*=1.2
        if crop in ["grapes","banana","papaya"] and potassium>100: boosted[i]*=1.2
    
    norm = boosted / boosted.sum()
    top5_idx = np.argsort(norm)[::-1][:5]
    return [(enc["crops"][i], round(float(norm[i])*100, 1)) for i in top5_idx]


# Test predictions
test_cases = [
    ("Nashik",  50, 60, 100, 22, 65, 6.2,  900),
    ("Nagpur",  60, 40,  80, 30, 60, 6.5,  750),
    ("Kolhapur",80, 50,  70, 26, 80, 6.8, 1100),
    ("Solapur", 40, 30,  50, 32, 45, 7.0,  400),
    ("Amravati",70, 55,  90, 29, 65, 6.3,  650),
]

print(f"{'District':<12} {'N':>4} {'P':>4} {'K':>4} {'Temp':>5} → {'Recommendation'}")
print("-"*70)
for case in test_cases:
    top5 = predict_crop(*case)
    print(f"{case[0]:<12} {case[1]:>4} {case[2]:>4} {case[3]:>4} {case[4]:>5}°C → {top5[0][0]} ({top5[0][1]}%)")
    print(f"   Top5: {[f"{c}({p}%)" for c,p in top5]}")

---
## 14. 📊 Additional Visualizations

In [ ]:
# Rainfall vs Humidity scatter
fig, ax = plt.subplots(figsize=(12, 7))
crops_sample = df_clean["crop_label"].unique()[:8]
palette = sns.color_palette("Set2", n_colors=8)
for crop, color in zip(crops_sample, palette):
    sub = df_clean[df_clean["crop_label"]==crop]
    ax.scatter(sub["rainfall"], sub["humidity"], alpha=0.5, label=crop, color=color, s=40)
ax.set_title("Rainfall vs Humidity by Crop Type", fontsize=14, fontweight="bold")
ax.set_xlabel("Rainfall (mm)")
ax.set_ylabel("Humidity (%)")
ax.legend(loc="upper right", ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature distribution
fig, ax = plt.subplots(figsize=(12, 5))
for crop in ["rice","cotton","grapes","orange","banana"]:
    sub = df_clean[df_clean["crop_label"]==crop]
    ax.hist(sub["temperature"], bins=20, alpha=0.6, label=crop)
ax.set_title("Temperature Distribution by Crop", fontsize=14, fontweight="bold")
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Nutrient balance radar chart
from matplotlib.patches import FancyArrowPatch
import matplotlib
categories_r = ["Nitrogen","Phosphorus","Potassium","pH","Humidity","Rainfall_scaled"]
crops_radar  = ["rice","cotton","grapes","banana","maize"]
N = len(categories_r)
angles = [n/float(N)*2*np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8,8), subplot_kw={"polar":True})
colors = ["#2e7d32","#0288d1","#ad1457","#ff8f00","#6a1b9a"]
for crop, color in zip(crops_radar, colors):
    sub = df_clean[df_clean["crop_label"]==crop]
    vals = [
        sub["nitrogen"].mean()/140,
        sub["phosphorus"].mean()/145,
        sub["potassium"].mean()/200,
        (sub["ph"].mean()-5.5)/(7.5-5.5),
        sub["humidity"].mean()/100,
        sub["rainfall"].mean()/1200,
    ]
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2, label=crop, color=color)
    ax.fill(angles, vals, alpha=0.15, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories_r, fontsize=11)
ax.set_title("Crop Nutrient & Climate Profile", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

---
## 15. 📋 Summary Report

In [ ]:
print("=" * 65)
print("  MAHARASHTRA CROP RECOMMENDATION SYSTEM — SUMMARY REPORT")
print("=" * 65)
print(f"
📁 Dataset")
print(f"   Total records  : {len(df_clean)}")
print(f"   Districts       : {df_clean["district_area_maharashtra"].nunique()}")
print(f"   Crop types      : {df_clean["crop_label"].nunique()}")
print(f"   Features        : {len(features)}")
print(f"
🤖 ML Models Trained")
for name in trained.keys():
    print(f"   ✓ {name}")
print(f"
🏆 Best Model")
print(f"   Model     : {best_final_name}")
print(f"   Accuracy  : {final_results[best_final_name]*100:.2f}%")
print(f"
📌 Feature Importance (from best model)")
if hasattr(best_final_model, "feature_importances_"):
    for f, imp in sorted(zip(feature_labels, best_final_model.feature_importances_), key=lambda x:-x[1]):
        bar = "█" * int(imp*100)
        print(f"   {f:<15} {imp:.4f}  {bar}")
print(f"
💾 Saved Files")
print(f"   crop_model.pkl  — Best ML model")
print(f"   encoders.pkl    — Encoders, scalers, metadata")
print(f"
🌐 Flask Application")
print(f"   Run: python app.py")
print(f"   URL: http://localhost:5000")
print("=" * 65)

---

## ✅ Notebook Complete!

This notebook covers:
- Full EDA with charts and heatmaps
- 5 ML model training and comparison
- Cross-validation and GridSearchCV
- Confusion matrix analysis
- Feature importance visualization
- Prediction function with hybrid ML + agronomy rules
- Saving model artifacts for Flask deployment

**Run  to launch the web application.**